In [84]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [ ]:
micro = pd.read_csv("dataset/microbiology_cohort_deid_tj_updated.csv")

C:\Users\nihal\AppData\Local\Temp\ipykernel_8572\3773327876.py:1: DtypeWarning: Columns (0: prelim_AST) have mixed types. Specify dtype option on import or set low_memory=False.
  micro = pd.read_csv("dataset/microbiology_cohort_deid_tj_updated.csv")


In [ ]:
combined_dataset = micro[["anon_id","CLSI_2022_pheno","order_time_jittered_utc_shifted"]]
combined_dataset["resistant"] = combined_dataset["CLSI_2022_pheno"] == "Resistant"


# 1. Track the original order of IDs as they first appear
original_order = combined_dataset['anon_id'].unique()

# 2. Sort by Flag (Descending: True comes before False)
# This pushes the preferred rows to the top for each ID group
df_sorted = combined_dataset.sort_values(by='resistant', ascending=False)

# 3. Drop duplicates based on ID, keeping the first occurrence
combined_dataset = df_sorted.drop_duplicates(subset=['anon_id'], keep='first')

# 4. Restore the original ID order
combined_dataset = combined_dataset.set_index('anon_id').loc[original_order].reset_index()


In [ ]:
print(combined_dataset)

        anon_id CLSI_2022_pheno order_time_jittered_utc_shifted  resistant
0         38476             NaN                      2062-11-30      False
1        108634             NaN                      2017-04-04      False
2        111803       Resistant                      1939-04-16       True
3         14459             NaN                      2037-07-08      False
4          2317             NaN                      1958-12-10      False
...         ...             ...                             ...        ...
226654   119880       Resistant                      1979-01-14       True
226655   103695     Susceptible                      2049-11-08      False
226656   131933             NaN                      2024-08-26      False
226657   146868             NaN                      2040-07-23      False
226658   121959     Susceptible                      1943-05-18      False

[226659 rows x 4 columns]


In [ ]:
depriv = pd.read_csv("dataset/ADI_deid_tj.csv")[["anon_id","adi_score","order_time_jittered_utc_shifted"]]


In [ ]:
combined_dataset = depriv.merge(combined_dataset, on=["anon_id","order_time_jittered_utc_shifted"], how="inner").drop_duplicates(subset=["anon_id"],keep="first").reset_index(drop=True)

In [ ]:
combined_dataset = combined_dataset[["anon_id","order_time_jittered_utc_shifted","resistant","adi_score"]]

In [ ]:
print(combined_dataset)

        anon_id order_time_jittered_utc_shifted  resistant  adi_score
0         38476                      2062-11-30      False  22.227193
1        108634                      2017-04-04      False  43.942857
2        111803                      1939-04-16       True        NaN
3         14459                      2037-07-08      False  12.125999
4          2317                      1958-12-10      False  22.186550
...         ...                             ...        ...        ...
226654   119880                      1979-01-14       True        NaN
226655   103695                      2049-11-08      False        NaN
226656   131933                      2024-08-26      False        NaN
226657   146868                      2040-07-23      False        NaN
226658   121959                      1943-05-18      False        NaN

[226659 rows x 4 columns]


In [ ]:
combined_dataset.to_csv("dataset/combined_dataset.csv")

In [2]:
comorb = pd.read_csv("dataset/comorbidity_deid_tj.csv")

In [3]:
comorb = comorb.drop_duplicates(subset=["anon_id","category"])
labels_list = list(comorb["category"].unique())

In [ ]:
comorbidities_enc = LabelEncoder()
comorbidities_enc.fit(comorb["category"])
print(len(comorbidities_enc.classes_))

508


In [24]:
print(len(comorb["anon_id"].unique()))

226659


In [15]:
print(None in list(comorb["category"].unique()))

False


In [28]:
labels_list = list(comorb["category"].unique())

comorb['Exists'] = True

# 2. Group by ID and Label, then unstack to create the columns
# We drop rows where Label is null/missing before grouping to avoid errors
df_pivoted = (comorb.dropna(subset=['category'])
              .groupby(['anon_id', 'category'], sort=False)['Exists']
              .any()
              .unstack(fill_value=False))

# 3. Filter out any columns that aren't in your master list of 508 labels
valid_columns = [col for col in df_pivoted.columns if col in labels_list]
df_pivoted = df_pivoted[valid_columns]

# 4. Use a safe dictionary alignment to add missing master labels as False
for label in labels_list:
    if label not in df_pivoted.columns:
        df_pivoted[label] = False

# 5. Restore original ID order safely using the original list sequence
original_id_order = [uid for uid in comorb['anon_id'].unique() if uid in df_pivoted.index]
df_final = df_pivoted.loc[original_id_order].reset_index()
comorb_final = df_final.drop(columns="category")

KeyError: "['category'] not found in axis"

In [29]:
print(df_final.columns)

Index(['anon_id', 'Blood loss anemia', 'Congestive heart failure',
       'Hypertension, complicated', 'Fluid and electrolyte disorders',
       'Other neurological disorders', 'Nutritional anemia',
       'Disorders of lipid metabolism', 'Other specified status',
       'Anxiety and fear-related disorders',
       ...
       'Traumatic arthropathy', 'Inhalant-related disorders',
       'Fetal alcohol syndrome',
       'Sequela of malnutrition and other nutritional deficiencies',
       'Liveborn', 'Hemolytic jaundice and perinatal jaundice',
       'Hemorrhagic and hematologic disorders of newborn', 'Birth trauma',
       'Amputation of a limb, subsequent encounter', nan],
      dtype='str', name='category', length=509)


In [ ]:
print(comorb[comorb["anon_id"] == 38476]["category"])

0                                     Blood loss anemia
1                              Congestive heart failure
5                             Hypertension, complicated
8                       Fluid and electrolyte disorders
9                          Other neurological disorders
10                                   Nutritional anemia
11                        Disorders of lipid metabolism
12                               Other specified status
13                   Anxiety and fear-related disorders
15                               Chronic kidney disease
16                      Malignant neuroendocrine tumors
18                                      Aplastic anemia
19    Conditions due to neoplasm or the treatment of...
20                  Diabetes mellitus with complication
24    Other specified and unspecified nutritional an...
28                                 Sleep wake disorders
29          Oculofacial plastics and orbital conditions
30                         Blindness and vision 

In [30]:
combined_dataset = pd.read_csv("dataset/combined_dataset.csv")

In [31]:
combined_dataset = combined_dataset.merge(df_final, on="anon_id", how="left")

In [35]:
combined_dataset = combined_dataset.drop(columns="Unnamed: 0")

In [36]:
print(combined_dataset.columns)

Index(['anon_id', 'order_time_jittered_utc_shifted', 'resistant', 'adi_score',
       'Blood loss anemia', 'Congestive heart failure',
       'Hypertension, complicated', 'Fluid and electrolyte disorders',
       'Other neurological disorders', 'Nutritional anemia',
       ...
       'Traumatic arthropathy', 'Inhalant-related disorders',
       'Fetal alcohol syndrome',
       'Sequela of malnutrition and other nutritional deficiencies',
       'Liveborn', 'Hemolytic jaundice and perinatal jaundice',
       'Hemorrhagic and hematologic disorders of newborn', 'Birth trauma',
       'Amputation of a limb, subsequent encounter', nan],
      dtype='str', length=512)


In [37]:
combined_dataset.to_csv("dataset/combined_dataset.csv")

In [2]:
demo = pd.read_csv("dataset/demographics_deid_tj.csv")

In [5]:
demo = demo.drop_duplicates(subset=["anon_id"])

In [7]:
combined_dataset = pd.read_csv("dataset/combined_dataset.csv")

In [8]:
combined_dataset = combined_dataset.merge(demo, how="left", on="anon_id")

In [14]:
combined_dataset = combined_dataset.drop(columns=["pat_enc_csn_id_coded", "order_proc_id_coded"])

In [15]:
combined_dataset.to_csv("dataset/combined_dataset.csv")

In [7]:
combined_dataset = pd.read_csv("dataset/combined_dataset.csv")

In [5]:
nursing_home = pd.read_csv("dataset/nursing_home_visits_deid_tj.csv")[["anon_id","nursing_home_visit_culture"]].drop_duplicates(subset="anon_id")

In [6]:
print(nursing_home)

        anon_id  nursing_home_visit_culture
0         38476                         NaN
1        108634                         NaN
2        111803                         NaN
3         14459                         NaN
4          2317                         NaN
...         ...                         ...
446132   119880                         NaN
446133   103695                         NaN
446138   131933                         NaN
446139   146868                         NaN
446140   121959                         NaN

[220142 rows x 2 columns]


In [ ]:
combined_dataset = combined_dataset.merge(nursing_home, how="left", on="anon_id").drop(columns="Unnamed: 0")

In [13]:
combined_dataset = combined_dataset.drop(columns="Unnamed: 512")

In [18]:
print(combined_dataset.columns)

Index(['Unnamed: 0', 'anon_id', 'order_time_jittered_utc_shifted', 'resistant',
       'adi_score', 'Blood loss anemia', 'Congestive heart failure',
       'Hypertension, complicated', 'Fluid and electrolyte disorders',
       'Other neurological disorders',
       ...
       'Fetal alcohol syndrome',
       'Sequela of malnutrition and other nutritional deficiencies',
       'Liveborn', 'Hemolytic jaundice and perinatal jaundice',
       'Hemorrhagic and hematologic disorders of newborn', 'Birth trauma',
       'Amputation of a limb, subsequent encounter', 'age', 'gender',
       'nursing_home_visit_culture'],
      dtype='str', length=515)


In [16]:
combined_dataset.to_csv("dataset/combined_dataset.csv")

In [3]:
prior_abx = pd.read_csv("dataset/prior_abx_deid_tj.csv")[["anon_id","order_time_jittered_utc_shifted","last_dose_to_culture","drug_code"]]

In [4]:
combined_dataset = combined_dataset.merge(prior_abx, on=["anon_id","order_time_jittered_utc_shifted"], how="left").drop_duplicates(subset=["anon_id"],keep="first").reset_index(drop=True)

In [6]:
print(combined_dataset.columns)

Index(['Unnamed: 0', 'anon_id', 'order_time_jittered_utc_shifted', 'resistant',
       'adi_score', 'Blood loss anemia', 'Congestive heart failure',
       'Hypertension, complicated', 'Fluid and electrolyte disorders',
       'Other neurological disorders',
       ...
       'Liveborn', 'Hemolytic jaundice and perinatal jaundice',
       'Hemorrhagic and hematologic disorders of newborn', 'Birth trauma',
       'Amputation of a limb, subsequent encounter', 'age', 'gender',
       'nursing_home_visit_culture', 'last_dose_to_culture', 'drug_code'],
      dtype='str', length=517)


In [7]:
combined_dataset.drop(columns="Unnamed: 0").to_csv("dataset/combined_dataset.csv")

In [5]:
prior_proc = pd.read_csv("dataset/prior_procedures_deid_tj.csv")[["anon_id","procedure_description","procedure_days_culture"]].drop_duplicates(subset="anon_id",keep="last")

In [6]:
print(prior_proc)

        anon_id procedure_description  procedure_days_culture
2        109396              mechvent                     168
6         56650    surgical_procedure                     464
7        156082    surgical_procedure                     497
8        158797    surgical_procedure                     710
10       156657    surgical_procedure                      19
...         ...                   ...                     ...
453438    31552              mechvent                     331
453440   156731    surgical_procedure                      35
453460   224089              mechvent                     487
453461   168267    surgical_procedure                      57
453462   146868    surgical_procedure                     998

[35580 rows x 3 columns]


In [8]:
combined_dataset = combined_dataset.merge(prior_proc, how="left", on="anon_id").drop(columns="Unnamed: 0")

In [10]:
print(combined_dataset.columns)

Index(['anon_id', 'order_time_jittered_utc_shifted', 'resistant', 'adi_score',
       'Blood loss anemia', 'Congestive heart failure',
       'Hypertension, complicated', 'Fluid and electrolyte disorders',
       'Other neurological disorders', 'Nutritional anemia',
       ...
       'Hemorrhagic and hematologic disorders of newborn', 'Birth trauma',
       'Amputation of a limb, subsequent encounter', 'age', 'gender',
       'nursing_home_visit_culture', 'last_dose_to_culture', 'drug_code',
       'procedure_description', 'procedure_days_culture'],
      dtype='str', length=518)


In [17]:
combined_dataset.to_csv("dataset/combined_dataset.csv")

In [13]:
ward = pd.read_csv("dataset/ward_type_deid_tj.csv")[["anon_id","order_time_jittered_utc_shifted","hosp_ward_IP","hosp_ward_OP","hosp_ward_ER","hosp_ward_UC","hosp_ward_day_surg"]]

In [14]:
combined_dataset = combined_dataset.merge(ward, on=["anon_id","order_time_jittered_utc_shifted"], how="left").drop_duplicates(subset=["anon_id"],keep="first").reset_index(drop=True)

In [16]:
print(combined_dataset.columns)

Index(['anon_id', 'order_time_jittered_utc_shifted', 'resistant', 'adi_score',
       'Blood loss anemia', 'Congestive heart failure',
       'Hypertension, complicated', 'Fluid and electrolyte disorders',
       'Other neurological disorders', 'Nutritional anemia',
       ...
       'nursing_home_visit_culture', 'last_dose_to_culture', 'drug_code',
       'procedure_description', 'procedure_days_culture', 'hosp_ward_IP',
       'hosp_ward_OP', 'hosp_ward_ER', 'hosp_ward_UC', 'hosp_ward_day_surg'],
      dtype='str', length=523)


In [4]:
combined_dataset = pd.read_csv("dataset/combined_dataset.csv")


In [5]:
threshold = 0.10
many_missing_cols = combined_dataset.columns[combined_dataset.isna().mean() > threshold].tolist()

print("Columns with > 30% missing values:", many_missing_cols)

Columns with > 30% missing values: ['adi_score', 'nursing_home_visit_culture', 'last_dose_to_culture', 'drug_code', 'procedure_description', 'procedure_days_culture']


In [ ]:
print(labels_list)

['Blood loss anemia', 'Congestive heart failure', 'Hypertension, complicated', 'Fluid and electrolyte disorders', 'Other neurological disorders', 'Nutritional anemia', 'Disorders of lipid metabolism', 'Other specified status', 'Anxiety and fear-related disorders', 'Chronic kidney disease', 'Malignant neuroendocrine tumors', 'Aplastic anemia', 'Conditions due to neoplasm or the treatment of neoplasm', 'Diabetes mellitus with complication', 'Other specified and unspecified nutritional and metabolic disorders', 'Sleep wake disorders', 'Oculofacial plastics and orbital conditions', 'Blindness and vision defects', 'Other specified eye disorders', 'Intestinal obstruction and ileus', 'Other specified and unspecified gastrointestinal disorders', 'Hepatic failure', 'Non-pressure ulcer of skin', 'Other specified joint disorders', 'Pathological fracture, initial encounter', 'Osteomyelitis', 'Acute and unspecified renal failure', 'Respiratory signs and symptoms', 'Symptoms of mental and substance 

In [12]:
labels_list_remove_nan = [
    col for col in labels_list
    if col is not None 
    and pd.notna(col) 
    and str(col).strip().lower() != 'nan'
]

In [2]:
comorb_bit_strings = combined_dataset[labels_list_remove_nan].fillna(0).astype(bool).astype(int).astype(str).agg(''.join,axis=1)
combined_dataset["comorbidities_int"] = comorb_bit_strings.apply(lambda x: int(x,2))

combined_dataset_reduced_comorbs = combined_dataset.drop(columns=labels_list_remove_nan)

combined_dataset_reduced_comorbs.to_csv("dataset/combined_datset_reduced_comorbs.csv")

import os
os.system("\a")

NameError: name 'combined_dataset' is not defined

In [ ]:
print(len(combined_dataset_reduced_comorbs.columns))

In [4]:
combined_dataset_reduced_comorbs = pd.read_csv("dataset/combined_datset_reduced_comorbs.csv")

In [5]:
combined_dataset_reduced_comorbs_no_nan = combined_dataset_reduced_comorbs.dropna(thresh=15).reset_index(drop=True)

In [6]:
combined_dataset_reduced_comorbs_no_nan.drop(columns="Unnamed: 0").to_csv("dataset/combined_dataset_reduced_comorbs_no_nan.csv")

In [7]:
print(len(combined_dataset_reduced_comorbs_no_nan), len(combined_dataset_reduced_comorbs))

167410 226659


In [5]:
combined_dataset_reduced_comorbs_no_nan = pd.read_csv("dataset/combined_dataset.csv")
labels_list = ["hosp_ward_IP","hosp_ward_OP","hosp_ward_ER","hosp_ward_UC","hosp_ward_day_surg"]

ward_bit_strings = combined_dataset_reduced_comorbs_no_nan[labels_list].fillna(0).astype(bool).astype(int).astype(str).agg(''.join,axis=1)
combined_dataset_reduced_comorbs_no_nan["wards_int"] = ward_bit_strings.apply(lambda x: int(x,2))

combined_dataset_reduced_comorbs_reduced_wards = combined_dataset_reduced_comorbs_no_nan.drop(columns=labels_list)

combined_dataset_reduced_comorbs_reduced_wards.to_csv("dataset/combined_dataset_reduced_wards.csv")

C:\Users\nihal\AppData\Local\Temp\ipykernel_12184\3069002955.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  combined_dataset_reduced_comorbs_no_nan["wards_int"] = ward_bit_strings.apply(lambda x: int(x,2))


In [25]:
print(combined_dataset.isna().sum().sort_values(ascending=False))

nursing_home_visit_culture         223875
procedure_description              191079
procedure_days_culture             191079
adi_score                          102720
last_dose_to_culture                62316
                                    ...  
order_time_jittered_utc_shifted         0
anon_id                                 0
Unnamed: 0                              0
Unnamed: 0.1                            0
wards_int                               0
Length: 521, dtype: int64


In [32]:
combined_dataset = pd.read_csv("dataset/combined_dataset_reduced_wards.csv")

In [33]:
combined_dataset_nurs_proc_flags = combined_dataset.drop(columns=['Unnamed: 0', 'Unnamed: 0.1']).reset_index(drop=True)
combined_dataset_nurs_proc_flags["nursing_home_visit_culture"] = combined_dataset_nurs_proc_flags["nursing_home_visit_culture"].notna()
combined_dataset_nurs_proc_flags["procedure_description"] = combined_dataset_nurs_proc_flags["procedure_description"].notna()

In [38]:
combined_dataset_nurs_proc_flags.to_csv("dataset/combined_dataset_reduced_wards_nurs_proc_flags.csv")

In [40]:
print(combined_dataset.columns)

Index(['Unnamed: 0', 'anon_id', 'order_time_jittered_utc_shifted', 'resistant',
       'adi_score', 'Blood loss anemia', 'Congestive heart failure',
       'Hypertension, complicated', 'Fluid and electrolyte disorders',
       'Other neurological disorders',
       ...
       'Birth trauma', 'Amputation of a limb, subsequent encounter', 'age',
       'gender', 'nursing_home_visit_culture', 'last_dose_to_culture',
       'drug_code', 'procedure_description', 'procedure_days_culture',
       'wards_int'],
      dtype='str', length=520)


In [53]:
combined_dataset = pd.read_csv("dataset/combined_dataset_reduced_wards.csv")

In [54]:
np_comorbs = combined_dataset.drop(columns=['Unnamed: 0', 'anon_id',
       'order_time_jittered_utc_shifted', 'resistant', 'adi_score','age',
       'gender', 'nursing_home_visit_culture', 'last_dose_to_culture',
       'drug_code', 'procedure_description', 'procedure_days_culture',
       'wards_int']).to_numpy().astype(bool)

In [16]:
print(np.isnan(np_comorbs).any())

False


In [55]:
combined_dataset = combined_dataset.drop(columns=['Unnamed: 0'])

In [56]:
comorb_cols = combined_dataset.drop(columns=['anon_id',
       'order_time_jittered_utc_shifted', 'resistant', 'adi_score','age',
       'gender', 'nursing_home_visit_culture', 'last_dose_to_culture',
       'drug_code', 'procedure_description', 'procedure_days_culture',
       'wards_int']).columns

In [57]:
from sklearn.decomposition import PCA

pca = PCA(n_components=15)

comorbs_reduced = pca.fit_transform(np_comorbs)

print(np.sum(pca.explained_variance_ratio_))

0.819202503870941


In [58]:
col_names = [f"Comorbidity_Reduced_Val_{i+1}" for i in range(comorbs_reduced.shape[1])]

comorbs_reduced_df = pd.DataFrame(comorbs_reduced, columns=col_names)

combined_dataset = combined_dataset.drop(columns=comorb_cols)
combined_dataset = pd.concat([combined_dataset,comorbs_reduced_df], axis=1)

combined_dataset.to_csv("dataset/combined_dataset_15comorbdims_1wardcode.csv")

In [31]:
combined_dataset = pd.read_csv("dataset/combined_dataset_11comorbdims_1wardcode.csv")

In [32]:
print(combined_dataset.columns)

Index(['Unnamed: 0', 'anon_id', 'order_time_jittered_utc_shifted', 'resistant',
       'adi_score', 'age', 'gender', 'nursing_home_visit_culture',
       'last_dose_to_culture', 'drug_code', 'procedure_description',
       'procedure_days_culture', 'wards_int', 'Comorbidity_Reduced_Val_1',
       'Comorbidity_Reduced_Val_2', 'Comorbidity_Reduced_Val_3',
       'Comorbidity_Reduced_Val_4', 'Comorbidity_Reduced_Val_5',
       'Comorbidity_Reduced_Val_6', 'Comorbidity_Reduced_Val_7',
       'Comorbidity_Reduced_Val_8', 'Comorbidity_Reduced_Val_9',
       'Comorbidity_Reduced_Val_10', 'Comorbidity_Reduced_Val_11'],
      dtype='str')


In [28]:
combined_dataset = combined_dataset.drop(columns="Unnamed: 0").reset_index(drop=True)

In [30]:
combined_dataset.to_csv("dataset/combined_dataset_11comorbdims_1wardcode.csv")

In [64]:
dataset = pd.read_csv("dataset/combined_dataset_reduced_wards.csv").dropna(subset="drug_code")

In [68]:
dataset.to_csv("dataset/dataset_no_nan_drug_code.csv")

In [69]:
dataset = pd.read_csv("dataset/dataset_no_nan_drug_code.csv")

In [70]:
antibiotics = dataset["drug_code"].unique()

In [77]:
print(antibiotics)

<StringArray>
['TZP', 'VAN', 'FLC', 'SXT', 'CRO', 'AMC', 'MTZ', 'SAM', 'CIP', 'AMX', 'NIT',
 'CPD', 'FOX', 'DOX', 'AZM', 'FEP', 'CFZ', 'CLI', 'MIN', 'CLR', 'LEX', 'CTT',
 'GEN', 'LVX', 'CFR', 'PEN', 'CAZ', 'CFM', 'FOF', 'MXF', 'TET', 'POS', 'CXM',
 'OXA', 'CDR', 'AMP', 'ATM', 'DCX', 'ITC', 'MEM', 'ETP', 'MFG', 'LZD', 'CPT',
 'TMP', 'IPM', 'VRC', 'DAP', 'NAF', 'CST', 'CTX', 'AMK', 'TOB', 'AMB', 'CZA',
 'TGC', 'TZD', 'C/T']
Length: 58, dtype: str


In [73]:
with open("dataset/metadata/drug_counts.txt", "w+") as file:
    file.write(str(dataset["drug_code"].value_counts()))

drug_code
CRO    18764
CFZ    15992
AZM    10905
DOX    10282
MTZ    10059
CIP     9994
SXT     9323
LEX     8954
NIT     8046
AMC     7879
TZP     6505
FEP     6467
FLC     5750
LVX     5019
AMX     5012
VAN     4926
CLI     3130
PEN     2657
SAM     2145
GEN     2100
FOF     1425
CPD     1258
FOX      935
CAZ      894
CXM      807
MIN      670
CLR      652
MEM      532
CFR      531
AMP      429
DCX      341
CDR      300
CTT      279
LZD      173
ETP      147
MFG      128
MXF      112
VRC      108
DAP       90
OXA       84
TMP       75
NAF       70
ITC       61
ATM       56
CFM       55
TET       53
IPM       45
POS       34
TOB       20
CPT       19
AMK       18
AMB       14
CTX        6
CST        4
TZD        4
CZA        2
TGC        2
C/T        1
Name: count, dtype: int64


In [78]:
for antibiotic in antibiotics:
    drug_df = dataset[dataset["drug_code"] == antibiotic].drop(columns="drug_code")
    drug_df.reset_index(drop=True)
    drug_df.to_csv(f"dataset/by_antibiotic/{antibiotic.replace("/","")}.csv")


In [85]:
import os
for filename in os.listdir("dataset/by_antibiotic"):
   df = pd.read_csv(f"dataset/by_antibiotic/{filename}")
   if (len(df)) < 12:
      continue
   np_comorbs = df.drop(columns=['Unnamed: 0.3', 'Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0', 'anon_id',
      'order_time_jittered_utc_shifted', 'resistant', 'adi_score','age',
      'gender', 'nursing_home_visit_culture', 'last_dose_to_culture',
      'procedure_description', 'procedure_days_culture',
      'wards_int']).to_numpy().astype(bool)
   comorb_cols = df.drop(columns=['Unnamed: 0.3', 'Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0','anon_id',
      'order_time_jittered_utc_shifted', 'resistant', 'adi_score','age',
      'gender', 'nursing_home_visit_culture', 'last_dose_to_culture',
      'procedure_description', 'procedure_days_culture',
      'wards_int']).columns
   pca = PCA(n_components=12)

   comorbs_reduced = pca.fit_transform(np_comorbs)

   with open("dataset/metadata/pca_retained_information.txt", "a") as file:
      file.write(f"\n{filename.replace(".csv", "")}: {np.sum(pca.explained_variance_ratio_)}")
   
   col_names = [f"Comorbidity_Reduced_Val_{i+1}" for i in range(comorbs_reduced.shape[1])]

   comorbs_reduced_df = pd.DataFrame(comorbs_reduced, columns=col_names)

   df = df.drop(columns=comorb_cols)
   df = pd.concat([df,comorbs_reduced_df], axis=1)
   df = df.drop(columns=['Unnamed: 0.3', 'Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0']).reset_index(drop=True)
   df.to_csv(f"dataset/by_antibiotic_12comorbdims/{filename}")
   
    

C:\Users\nihal\AppData\Local\Temp\ipykernel_18748\2835880426.py:3: DtypeWarning: Columns (0: Blood loss anemia, 1: Congestive heart failure, 2: Hypertension, complicated, 3: Fluid and electrolyte disorders, 4: Other neurological disorders, 5: Nutritional anemia, 6: Disorders of lipid metabolism, 7: Other specified status, 8: Anxiety and fear-related disorders, 9: Chronic kidney disease, 10: Malignant neuroendocrine tumors, 11: Aplastic anemia, 12: Conditions due to neoplasm or the treatment of neoplasm, 13: Diabetes mellitus with complication, 14: Other specified and unspecified nutritional and metabolic disorders, 15: Sleep wake disorders, 16: Oculofacial plastics and orbital conditions, 17: Blindness and vision defects, 18: Other specified eye disorders, 19: Intestinal obstruction and ileus, 20: Other specified and unspecified gastrointestinal disorders, 21: Hepatic failure, 22: Non-pressure ulcer of skin, 23: Other specified joint disorders, 24: Pathological fracture, initial encoun